# 03 — Deep Agent Tracing with Microsoft Foundry

## Goal

Instrument our existing Deep Agent using OpenTelemetry and send its
execution traces to Azure Application Insights / Microsoft Foundry.

We are not changing the agent architecture.

Previous architecture:

```
User
  ↓
Deep Agent
  ↓
LangGraph
  ├── Deep Agent local tools
  └── Foundry model
        ↓
     Web Search
```

New architecture:

```
User
  ↓
Deep Agent
  ↓
LangGraph
  ├── local tools
  └── Foundry model + Web Search
          │
          │ telemetry
          ▼
     OpenTelemetry
          ↓
 Application Insights
          ↓
 Foundry / Azure Monitor traces
```

The purpose of tracing is not merely logging.

We want to understand the execution trajectory:

- model calls
- tool calls
- latency
- failures
- agent flow
- inputs and outputs

In [2]:
import os

from dotenv import load_dotenv

load_dotenv()

PROJECT_ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
MODEL_DEPLOYMENT = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]
APP_INSIGHTS_CONNECTION_STRING = os.environ[
    "APPLICATIONINSIGHTS_CONNECTION_STRING"
]

# print("Project:", PROJECT_ENDPOINT)
# print("Model:", MODEL_DEPLOYMENT)
# print("Application Insights configured:", bool(APP_INSIGHTS_CONNECTION_STRING))

In [3]:
from azure.identity import DefaultAzureCredential
from langchain_azure_ai.chat_models import AzureAIOpenAIApiChatModel

credential = DefaultAzureCredential()

model = AzureAIOpenAIApiChatModel(
    project_endpoint=PROJECT_ENDPOINT,
    credential=credential,
    model=MODEL_DEPLOYMENT,
)

print(type(model))

<class 'langchain_azure_ai.chat_models.openai.AzureAIOpenAIApiChatModel'>


## Recreate Deep Agent

In [4]:
from langchain_azure_ai.tools.builtin import WebSearchTool

web_search = WebSearchTool()

print(type(web_search))

<class 'langchain_azure_ai.tools.builtin._tools.WebSearchTool'>


C:\Users\shchitt\AppData\Local\Temp\ipykernel_9516\2030170112.py:3: ExperimentalWarning: WebSearchTool is currently in preview and is subject to change. This preview is provided without a service-level agreement, and we don't recommend it for production workloads. Certain features might not be supported or might have constrained capabilities. For more information, see https://azure.microsoft.com/support/legal/preview-supplemental-terms
  web_search = WebSearchTool()


In [5]:
research_instructions = """
You are an expert research assistant.

Your job is to research questions thoroughly and produce concise,
evidence-grounded answers.

## Research behavior

- Use web search whenever current or externally verifiable information is needed.
- Prefer authoritative and primary sources when possible.
- Search more than once when the first search does not fully answer the question.
- Distinguish established facts from interpretation.
- Include source citations in your final answer when web research was used.

You also have the standard Deep Agents capabilities for planning,
filesystem-based context management, and subagent delegation.
"""

In [6]:
from deepagents import create_deep_agent

research_agent = create_deep_agent(
    model=model,
    tools=[web_search],
    system_prompt=research_instructions,
)

print(type(research_agent))

<class 'langgraph.graph.state.CompiledStateGraph'>


# What is a tracer?

A tracer observes operations performed by the application.

A single request produces a **trace**.

The trace contains individual units of work called **spans**.

For example:

```
Research request                       ← trace
│
├── agent invocation                  ← span
│
├── model invocation                  ← span
│
├── web search                        ← span
│
├── write_file                        ← span
│
├── read_file                         ← span
│
└── final model invocation            ← span
```

Each span can contain information such as:

- start time
- end time
- duration
- operation name
- parent operation
- model name
- tool name
- status
- inputs/outputs
- token information

This gives us a structured representation of the agent trajectory.

## Create the OpenTelemetry tracer

In [7]:
from langchain_azure_ai.callbacks.tracers import AzureAIOpenTelemetryTracer

azure_tracer = AzureAIOpenTelemetryTracer(
    connection_string=APP_INSIGHTS_CONNECTION_STRING,
    enable_content_recording=True,
)

print(type(azure_tracer))

<class 'langchain_azure_ai.callbacks.tracers.inference_tracing.AzureAIOpenTelemetryTracer'>


In [9]:
traced_research_agent = research_agent.with_config(
    {
        "callbacks": [azure_tracer],
    }
)

print(type(traced_research_agent))

<class 'langgraph.graph.state.CompiledStateGraph'>


In [10]:
small_question = """
What is Microsoft Foundry Hosted Agents?

Use current Microsoft documentation and explain it in 3 concise bullets.
"""

small_result = traced_research_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": small_question,
            }
        ]
    }
)

In [11]:
print(small_result["messages"][-1].content)

[{'id': 'ws_0559c0fa76b5ffed006a9d149285e4819391dc1ab82a457851', 'action': {'type': 'search', 'queries': ['Microsoft Foundry Hosted Agents documentation', 'site:learn.microsoft.com "Foundry" "Hosted Agents"', '"Foundry Hosted Agents" Microsoft', 'Microsoft Foundry hosted agents Azure DevOps'], 'query': 'Microsoft Foundry Hosted Agents documentation'}, 'status': 'completed', 'type': 'web_search_call', 'response_id': 'resp_0559c0fa76b5ffed006a9d1491bb8481939e74b5498477bc21'}, {'type': 'text', 'text': '- **Managed hosting for your agent code (containers) in Foundry Agent Service:** Microsoft Foundry Hosted Agents let you deploy agents (including Agent Framework agents) as **containerized applications** onto **Microsoft-managed infrastructure**, so you don’t have to build and run the hosting stack yourself. ([learn.microsoft.com](https://learn.microsoft.com/en-us/agent-framework/hosting/foundry-hosted-agent))\n\n- **Platform handles the “production plumbing”:** The service takes care of **

In [12]:
def inspect_agent_activity(messages):
    for i, message in enumerate(messages):
        print(f"\n### Message {i}: {type(message).__name__}")

        tool_calls = getattr(message, "tool_calls", None)
        if tool_calls:
            print("LOCAL / AGENT TOOL CALLS:")
            for call in tool_calls:
                print(" ", call)

        content_blocks = getattr(message, "content_blocks", None)
        if content_blocks:
            print("CONTENT BLOCKS:")
            for block in content_blocks:
                print("  -", block.get("type"))
                
inspect_agent_activity(small_result["messages"])


### Message 0: HumanMessage
CONTENT BLOCKS:
  - text

### Message 1: AIMessage
CONTENT BLOCKS:
  - server_tool_call
  - server_tool_result
  - text


### Give the tracer an agent identity

In [13]:
named_tracer = AzureAIOpenTelemetryTracer(
    connection_string=APP_INSIGHTS_CONNECTION_STRING,
    enable_content_recording=True,
    agent_id="deep-agents-foundry-research",
)

print(type(named_tracer))

<class 'langchain_azure_ai.callbacks.tracers.inference_tracing.AzureAIOpenTelemetryTracer'>


In [14]:
import inspect

print(
    inspect.signature(AzureAIOpenTelemetryTracer)
)

(*, connection_string: 'Optional[str]' = None, enable_content_recording: 'bool' = True, project_endpoint: 'Optional[str]' = None, credential: 'Optional[Any]' = None, name: 'str' = 'AzureAIOpenTelemetryTracer', agent_id: 'Optional[str]' = None, provider_name: 'Optional[str]' = None, message_keys: 'Optional[Sequence[str]]' = None, message_paths: 'Optional[Sequence[str]]' = None, trace_all_langgraph_nodes: 'bool' = False, ignore_start_node: 'bool' = True, compat_create_agent_filtering: 'bool' = True, auto_configure_azure_monitor: 'Optional[bool]' = None, trace_state: 'Optional[bool]' = None, max_state_size: 'Optional[int]' = None, _prepare_messages_fn: 'Optional[Callable[..., tuple[Optional[str], Optional[str]]]]' = None) -> 'None'


In [15]:
traced_research_agent = research_agent.with_config(
    {
        "callbacks": [named_tracer],
    }
)

In [16]:
filesystem_question = """
Research the current Microsoft guidance for building and operating
custom agents in Foundry.

Do the following:

1. Research the topic from current Microsoft documentation.
2. Organize your important findings in /research_notes.md.
3. Read the notes back.
4. Produce a concise final architecture summary with citations.

Do not simply answer from memory.
"""

filesystem_result = traced_research_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": filesystem_question,
            }
        ]
    }
)

In [17]:
inspect_agent_activity(filesystem_result["messages"])


### Message 0: HumanMessage
CONTENT BLOCKS:
  - text

### Message 1: AIMessage
LOCAL / AGENT TOOL CALLS:
  {'name': 'write_file', 'args': {'file_path': '/research_notes.md', 'content': '# Microsoft guidance: building & operating *custom agents* in Microsoft Foundry (2026-09-06)\n\n## Scope & terminology\n- **Foundry Agent Service**: Managed platform to build, deploy, and scale agents. Supports using *any framework* + models from the **Foundry model catalog**, with a single entry point for model inference and tools. \ue200cite\ue202turn0search1\ue201\n- **Hosted agent**: Your custom/containerized agent code runs on Foundry Agent Service; the agent calls Foundry models for reasoning while your code handles orchestration. Foundry provides secure/scalable operations. \ue200cite\ue202turn0search7\ue202turn0search8\ue201\n- **Custom agent (Control Plane)**: An agent you deploy yourself (Azure compute or other cloud) and expose via a reachable endpoint; you can **register** it with Foundry C

# Local tools vs server-side tools in traces

A local Deep Agents tool such as:

write_file()

executes in our Python/LangGraph runtime.

So tracing can naturally look like:

agent
  ↓
tool span: write_file


Foundry Web Search is different.

It executes inside the Foundry model request:

agent
  ↓
model call
      ↓
  server-side web search


Therefore the exact span hierarchy may differ.

This is not necessarily missing telemetry.

It reflects where the operation executes.

### Measure the executime time from Python too

In [18]:
import time

start = time.perf_counter()

timed_result = traced_research_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """
Research the latest Microsoft guidance on evaluating AI agents.
Summarize the three most important recommendations.
""",
            }
        ]
    }
)

elapsed = time.perf_counter() - start

print(f"Total wall-clock time: {elapsed:.2f} seconds")

Total wall-clock time: 20.18 seconds


In [19]:
inspect_agent_activity(timed_result["messages"])


### Message 0: HumanMessage
CONTENT BLOCKS:
  - text

### Message 1: AIMessage
CONTENT BLOCKS:
  - server_tool_call
  - server_tool_result
  - server_tool_call
  - server_tool_result
  - text


## Inspect token metadata

In [20]:
final_message = timed_result["messages"][-1]

print("Usage metadata:")
print(getattr(final_message, "usage_metadata", None))

print("\nResponse metadata:")
print(getattr(final_message, "response_metadata", None))

Usage metadata:
{'input_tokens': 13245, 'output_tokens': 543, 'total_tokens': 13788, 'input_token_details': {'cache_creation': 0, 'cache_read': 6272}, 'output_token_details': {'reasoning': 137}}

Response metadata:
{'id': 'resp_027d732ae1cc3bd0006a9d16b84c148197a53c2389c437dc12', 'created_at': 1788679864.0, 'metadata': {}, 'model': 'gpt-5.2', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-5.2'}


## Run two tasks and commpare traces

In [21]:
simple_query = """
What is a Foundry Hosted Agent?
Answer in two sentences using current Microsoft documentation.
"""

simple_result = traced_research_agent.invoke(
    {"messages": [{"role": "user", "content": simple_query}]}
)

In [22]:
complex_query = """
Research three viable approaches for productionizing a custom
enterprise research agent on Microsoft Foundry.

Compare them across:

- orchestration control
- deployment
- identity
- tools
- observability
- evaluation
- operational complexity

Use current Microsoft documentation and recommend when each approach fits.
"""

complex_result = traced_research_agent.invoke(
    {"messages": [{"role": "user", "content": complex_query}]}
)

# Message history vs tracing

## Message inspection

`result["messages"]`

tells us primarily:

- what messages existed
- local tool calls
- tool results
- server-side content blocks
- final response


## Distributed tracing

OpenTelemetry gives us another dimension:

- operation hierarchy
- duration
- timing
- failures
- agent/model/tool identity
- model metadata
- token usage
- correlation across operations


## Together

Messages answer:

> What happened logically?

Tracing answers:

> How did the execution happen operationally?

# Tracing can expose sensitive information

We enabled:

`enable_content_recording=True`

because this is a learning environment.

That can record:

- user prompts
- model responses
- tool parameters
- tool results
- retrieved content


This is extremely useful for debugging.

It can also be dangerous in production.

Examples:

- customer data
- source code
- proprietary documents
- credentials accidentally passed through prompts
- PII
- regulated information


Production tracing requires a deliberate policy for:

- content recording
- sampling
- redaction
- access controls
- retention

## One Final Trajectory Experiment - Deep Research Question

In [23]:
deep_research_question = """
Investigate how I could run an open-source Deep Agents / LangGraph
research agent as a production workload on Microsoft Foundry.

Research and reason through:

1. model integration
2. web search/tools
3. Hosted Agent deployment
4. agent identity
5. workspace/state
6. observability
7. evaluation

Organize intermediate findings in the filesystem if useful.

Produce:

- recommended architecture
- major trade-offs
- current limitations
- implementation sequence

Use current Microsoft documentation.
"""

deep_research_result = traced_research_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": deep_research_question,
            }
        ]
    }
)

In [24]:
inspect_agent_activity(deep_research_result["messages"])


### Message 0: HumanMessage
CONTENT BLOCKS:
  - text

### Message 1: AIMessage
LOCAL / AGENT TOOL CALLS:
  {'name': 'write_file', 'args': {'file_path': '/mnt/data/foundry_research_notes.md', 'content': '# Running an open-source Deep Agents / LangGraph research agent as a production workload on Microsoft Foundry (notes)\n\nDate: 2026-09-07\n\n## Key Foundry primitives (for agent workloads)\n- **Foundry Agent Service**: platform that hosts agents, tools, memory, vector stores, and evaluation/observability.\n- **Hosted agents**: bring-your-own code/framework as a container image; Foundry manages sessions (stateful sandboxes), scaling, identity, tracing. Docs: Deploy a hosted agent; Hosted agents concepts. \ue200cite\ue202turn0search0\ue202turn1search6\ue201\n- **Session**: isolated sandbox + persistent filesystem; persists across turns and idle; sessions persist up to 30 days; idle timeout 5–60 min (default 15). \ue200cite\ue202turn2search1\ue201\n- **Agent identity**: Foundry provisions

In [26]:
from IPython.display import display, Markdown

display(Markdown(deep_research_result['messages'][-1].content[-1]['text']))

## Recommended architecture (production)

### A. Control plane (Foundry project + governed assets)
1. **Microsoft Foundry Project**
   - Create a Foundry project and choose **Basic** vs **Standard agent setup** depending on whether you need customer-managed resources and stronger isolation. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/environment-setup))
2. **Model deployments**
   - Deploy your chosen chat model(s) from the Foundry model catalog and treat them as the “reasoning engines” your LangGraph/Deep Agents code calls. Hosted agents are designed so *Foundry models do reasoning while your code orchestrates*. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents))
3. **Tooling endpoints**
   - Prefer **Foundry-managed tools** where possible:
     - **Web search tool** for public-web grounding with citations. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/web-search))  
     - **Azure AI Search tool** for enterprise/RAG grounding with citations. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/ai-search))  
     - **MCP tool** to connect to remote tools (your own or vendors) via Model Context Protocol. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/model-context-protocol))  
   - Optionally consolidate tools behind a single **Toolbox / MCP endpoint** to keep your agent code simple (one MCP URL, discover tools at runtime). ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/quickstarts/quickstart-toolbox-agent))

### B. Data/state plane (sessions + durable stores)
1. **Hosted Agent Sessions**
   - Run each workload in a Foundry **session** (stateful sandbox). Foundry **persists the session filesystem** (`$HOME` + uploaded files) across turns and idle periods; sessions persist up to **30 days**; idle timeout is configurable **5–60 min** (default **15**). ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/manage-hosted-sessions))  
2. **Long-term memory (optional)**
   - Use Foundry **Memory** (managed memory stores) if you want cross-session personalization / long-lived facts. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/quickstarts/quickstart-memory-hosted-agent))  
3. **Bring-your-own resources (compliance / ownership / networking)**
   - If you need CMK, full data ownership, or network isolation, configure Agent Service to use your own **Azure OpenAI, Storage, Cosmos DB, and Azure AI Search** resources (instead of Microsoft-managed defaults). ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/use-your-own-resources))  

### C. Runtime (your open-source agent code)
1. **Hosted Agent deployment**
   - Package your LangGraph/Deep Agents research agent as a container and **deploy as a Hosted Agent** to Foundry Agent Service; Foundry manages endpoint + versions + lifecycle. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/deploy-hosted-agent))  
2. **Invocation**
   - Your app calls the agent’s **stable endpoint**; a single endpoint can serve many users, while sessions keep each user/workload isolated. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/configure-agent))  

### D. Identity & security
1. **Agent identity (no embedded secrets)**
   - Foundry automatically provisions Microsoft Entra **agent identities** and uses them for governance and for authenticating to tools/downstream systems via RBAC (least privilege). ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/agent-identity))  
2. **Tool authentication**
   - For MCP servers and other tool endpoints, use the supported auth modes (key-based, Entra, OAuth) depending on sensitivity and publisher support. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/mcp-authentication))  

### E. Observability + evaluation loop
1. **Tracing**
   - Connect an **Application Insights** resource and enable tracing; tracing is **GA for prompt and hosted agents** (workflow/external agents are preview). ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/trace-agent-setup))  
   - For framework-level spans (LangGraph/LangChain), instrument with Foundry’s recommended OpenTelemetry/OpenInference approach. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/trace-agent-framework))  
2. **Evaluation**
   - Run agent-targeted evaluation using a **rubric evaluator generated from agent context**, plus built-in evaluators (safety, groundedness, etc.). ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/evaluate-agent))  
   - Convert production traces into evaluation datasets (preview) to continuously harden the agent based on real usage. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/traces-to-dataset))  
   - Use azd-driven evaluation workflows if you want a repeatable “quality gate” in CI/CD. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/azure-developer-cli-evaluation))  

---

## Research/Reasoning through your 7 requested areas

### 1) Model integration
**Pattern**
- In your containerized agent, call your Foundry model deployment as the LLM behind LangGraph nodes (planner, researcher, writer, critic).
- Keep model selection/versioning in Foundry; keep orchestration logic in code.

**Why this fits Foundry**
- Hosted agents are explicitly meant to let Foundry models handle reasoning while your custom code handles orchestration. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents))

### 2) Web search / tools
**Recommended**
- Use Foundry’s **web search tool** for real-time internet grounding. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/web-search))
- For enterprise corpora, use **Azure AI Search tool** or your own RAG tool exposed via MCP. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/ai-search))
- Standardize all “actions” as tools (web search, fetch, internal APIs, ticketing, storage) behind **MCP**; optionally front them with a Foundry toolbox so your agent only needs one MCP endpoint. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/quickstarts/quickstart-toolbox-agent))

### 3) Hosted Agent deployment
**What you deploy**
- Your open-source LangGraph/Deep Agents service as a container (or zip-source build if you adopt that flow), and register it as a Hosted Agent version in Foundry. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/deploy-hosted-agent))

**How production works**
- Each user/workload maps to a **session**, which provides stateful execution and persistent filesystem. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/manage-hosted-sessions))

### 4) Agent identity
- Foundry provisions an **Entra agent identity** to avoid secrets-in-code and enable governed access to downstream resources. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/agent-identity))
- Grant RBAC to the agent identity for the exact resources/tools it needs (Search index reader, Storage blob reader, Cosmos contributor, etc.).

### 5) Workspace / state
- Use **session filesystem** for short/medium-lived working artifacts (downloaded pages, intermediate JSON, cached embeddings, scratchpads). Foundry persists `$HOME` across turns and idle and retains sessions up to 30 days. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/manage-hosted-sessions))  
- Use **Memory stores** for durable user preferences or “learned facts” across sessions. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/quickstarts/quickstart-memory-hosted-agent))  
- Use **BYO resources** if you need stronger controls (CMK, isolation, ownership) for conversations/files/vector stores. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/use-your-own-resources))

### 6) Observability
- Turn on server-side tracing by connecting Application Insights; it captures latency, exceptions, prompt content, retrieval ops, etc. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/trace-agent-setup))  
- Add framework instrumentation (LangGraph/LangChain) using Foundry’s tracing guidance. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/trace-agent-framework))

### 7) Evaluation
- Use the Foundry agent evaluation flow: rubric evaluator from your agent context + built-in evaluators, run against a dataset. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/evaluate-agent))  
- Close the loop by generating datasets from production traces (preview) and re-running evaluations. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/traces-to-dataset))  
- Operationalize with azd evaluation recipes as gates in your delivery pipeline. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/azure-developer-cli-evaluation))

---

## Major trade-offs

1. **Foundry-managed tools vs BYO tools (MCP/your APIs)**
   - Managed tools (web search, AI Search integration) are fastest to production and align with Foundry governance/telemetry. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/web-search))  
   - BYO MCP tools give maximum flexibility and portability, but you now own tool reliability, auth configuration, and potentially more observability work. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/model-context-protocol))

2. **Session filesystem vs external durable storage**
   - Session FS is great for “agent scratch space” and makes LangGraph state management easy, but it’s scoped to the session lifecycle (up to 30 days) and isn’t a general data lake. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/manage-hosted-sessions))  
   - External stores (Storage/Cosmos/Search) provide durability and enterprise governance but require schema, lifecycle, and cost management. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/use-your-own-resources))

3. **Basic vs Standard setup**
   - Basic is quicker; Standard emphasizes single-tenant/customer-managed resources for stronger control and isolation. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/standard-agent-setup))

4. **“Open-source agent framework purity” vs platform alignment**
   - You can run LangGraph largely unchanged, but you’ll get more operational leverage by aligning with Foundry primitives (sessions, tool catalog/MCP, agent identity, evaluation/tracing).

---

## Current limitations / watch-outs (from current docs)

- **Tracing coverage**: Tracing is GA for prompt + hosted agents; **workflow and external agents are preview** (avoid depending on preview-only parts for critical production commitments). ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/trace-agent-setup))  
- **BYO resources**: Supported, but the docs explicitly call out **limitations**—plan time to validate feature parity and networking constraints in your environment. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/use-your-own-resources))  
- **Session lifecycle constraints**: Sessions persist up to **30 days**, and idle timeout behavior affects “always-on” assumptions (compute is deprovisioned after idle). ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/manage-hosted-sessions))  
- **Legacy vs new publishing**: Some documentation references legacy “Agent Applications” and migration to the newer stable endpoint model—avoid building new systems on the legacy publishing flow. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/agent-applications))  

---

## Implementation sequence (practical, low-risk)

1. **Foundation**
   - Create Foundry project; pick **Basic** vs **Standard** setup. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/environment-setup))  
   - Deploy model(s) you intend to use.

2. **Prototype the agent in a container locally**
   - Implement LangGraph/Deep Agents workflow with an abstraction layer:
     - LLM client = Foundry model endpoint
     - Tools = (a) Foundry web search, (b) MCP toolbox endpoint, (c) optional AI Search tool
   - Write state to local FS paths that will map cleanly to session `$HOME`.

3. **Deploy as Hosted Agent (dev)**
   - Deploy container as a **Hosted Agent**; create a version; validate basic invocation and session behavior. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/deploy-hosted-agent))  
   - Use the sessions API patterns; confirm idle timeout + resume works for your workload. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/manage-hosted-sessions))

4. **Identity hardening**
   - Identify downstream resources (Search, Storage, internal APIs) and grant least-privilege RBAC to the **agent identity**. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/agent-identity))  
   - If using MCP tools, configure auth mode (Entra/OAuth/key) and validate end-to-end. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/mcp-authentication))

5. **Observability**
   - Connect Application Insights; enable server-side tracing; add framework instrumentation if needed. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/trace-agent-setup))  

6. **Evaluation baseline**
   - Create a small evaluation dataset (20–100 cases), generate a rubric evaluator from agent context, add safety/groundedness evaluators, run evaluations. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/evaluate-agent))  
   - Decide thresholds for deployment promotion.

7. **Production readiness**
   - Add trace-to-dataset generation (if you’re comfortable with preview) to keep tests aligned with real traffic. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/traces-to-dataset))  
   - Build CI/CD with versioned agent deployments + azd evaluation recipe as a gate. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/azure-developer-cli-evaluation))  

---

### If you want, I can tailor this to your exact “Deep Agents / LangGraph research agent” shape
Answer these and I’ll map them to concrete Foundry choices:
1) Do you need private network egress (VNet) to reach internal endpoints?  
2) Do you need CMK / “data stays in my subscription” (Standard + BYO resources)?  
3) What tool set do you need (web, internal APIs, file ingestion, ticketing, GitHub, etc.)—should we front it with MCP/toolbox?

# What we learned

## 1. We did not modify the Deep Agent

The same agent from notebook 02 now emits telemetry.

Deep Agent
   ↓
LangGraph
   ↓
AzureAIOpenTelemetryTracer
   ↓
OpenTelemetry
   ↓
Application Insights


## 2. A trace represents the entire agent run

Trace
├── agent invocation
├── model operations
├── tools
└── other child operations

Individual operations are spans.


## 3. Tracing and message history are complementary

Messages explain logical state transitions.

Tracing explains operational execution.


## 4. Local and server-side tools may look different

Deep Agents filesystem tools execute locally through LangGraph.

Foundry Web Search executes inside the Foundry model request.

Their telemetry structures therefore differ.


## 5. Observability is more than debugging

Tracing gives us raw material for:

- latency analysis
- token economics
- failure analysis
- tool-use analysis
- trajectory evaluation
- regression detection


## 6. Current architecture

                       User
                         ↓
                    Deep Agent
                         ↓
                     LangGraph
              ┌──────────┴──────────┐
              │                     │
       local agent tools        Foundry model
              │                     │
       write/read/etc.         Web Search
              │                     │
              └──────────┬──────────┘
                         │
                  OpenTelemetry
                         ↓
               Application Insights
                         ↓
                  Foundry / Azure
                    Observability


## Next milestone

Notebook 04 will move from:

"What did the agent do?"

to:

"Was what the agent did actually good?"

We will evaluate:

- final-answer quality
- grounding
- task completion
- tool usage
- trajectory
- efficiency

using deterministic checks plus Foundry evaluators and rubric-based evaluation.